# Домашнее задание № 3. Исправление опечаток

## 1. Учет грамматики при оценке исправлений (3 балла)

В последнюю итерацию алгоритма для генерации исправлений добавьте еще один компонент - учет грамматической информации. Частично она уже учитывается за счет языковой модели (вероятность предсказывается для словоформы), но такой подход ограничен из-за того, что модель не может ничего предсказать для словоформ, которых не было в обучающей выборке. Чтобы это исправить постройте еще одну "языковую модель" на грамматических тэгах:
1) Используя mystem или pymorphy, разметьте какой-нибудь корпус (например, кусок wiki из семинара) или воспользуйтесь уже размеченным корпусом (например, opencorpora)
2) соберите униграмные и биграмные статистики на уровне грамматических тэгов (например, вместо `задача важна` у вас будет биграм `S,жен,неод=им,ед A=ед,кр,жен`). Для простоты можете начать только с частеречных тэгов и добавить остальную информацию позже
3) напишите функцию, которая будет оценивать вероятность данного предложения на основе грамматической языковой модели (статистик из предыдущего шага). Функция должна сначала преобразовать текст в грамматические тэги, используя точно такой же подход, что использовался на шаге 1.
4) в функции correct_text_with_lm замените compute_sentence_proba на вашу новую функцию и прогоните получившийся алгоритм на данных
5) сравните предсказания с предсказанием изначального correct_text_with_lm, проверьте метрики и посмотрите на различие в ошибках и исправлениях, найдите несколько примеров отличий в предсказаниях этих подходов

In [ ]:
!pip install pymorphy2

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 32.6 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=48c66bc5b5c161fb2a5cbb27c314c5809be34aa25e9bf6ea853bf9db92cb6be3
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


In [ ]:
corpus = open('data/wiki_data.txt', encoding='utf8').read()

## 2.  Symspell (5 баллов)

Реализуйте алгоритм Symspell. Он похож на алгоритм Норвига, но проще и быстрее. Он основан только на одной операции - удалении символа. Описание алгоритма по шагам:

1) Составляется словарь правильных слов  
2) На основе словаря правильных слов составляется словарь удалений - для каждого правильного слова создаются все варианты удалений и создается словарь, где ключ - слово с удалением, а значение - правильное слово  (обратите внимание, что для одного удаления может быть несколько правильных слов!)
3) При исправлении слова с опечаткой сначала само слово проверятся по словарю удаления, а затем для этого слова генерируются все варианты удалений, и каждый вариант проверяется по словарю удалений. Если в словаре удалений таким образом находится совпадение, то соответствующее ему правильное слово становится исправлением.
Если совпадений несколько, то выбирается наиболее вероятное правильное слово  


Оцените качество полученного алгоритма теми же тремя метриками.

In [ ]:
import re
from collections import Counter, defaultdict
import time

class SymSpell:
    def __init__(self, max_edit_distance=2):
        self.max_edit_distance = max_edit_distance
        self.dictionary = Counter()
        self.deletions_dict = defaultdict(list)

    def build_dictionary(self, words):
        """Строит словарь правильных слов и словарь удалений"""
        self.dictionary = Counter(words)

        # Строим словарь удалений
        self.deletions_dict = defaultdict(list)
        for word, freq in self.dictionary.items():
            # Добавляем само слово как кандидата для самого себя
            self.deletions_dict[word].append((word, 0, freq))

            # Генерируем все варианты удалений
            for i in range(len(word)):
                deletion = word[:i] + word[i+1:]
                self.deletions_dict[deletion].append((word, 1, freq))

                # Генерируем удаления второго порядка (если нужно)
                if self.max_edit_distance >= 2:
                    for j in range(len(deletion)):
                        deletion2 = deletion[:j] + deletion[j+1:]
                        self.deletions_dict[deletion2].append((word, 2, freq))

    def generate_deletions(self, word, max_distance):
        """Генерирует все варианты удалений для слова"""
        deletions = set()
        deletions.add(word)  # само слово

        # Удаления первого порядка
        for i in range(len(word)):
            deletion = word[:i] + word[i+1:]
            deletions.add(deletion)

            # Удаления второго порядка
            if max_distance >= 2:
                for j in range(len(deletion)):
                    deletion2 = deletion[:j] + deletion[j+1:]
                    deletions.add(deletion2)

                    # Удаления третьего порядка
                    if max_distance >= 3:
                        for k in range(len(deletion2)):
                            deletion3 = deletion2[:k] + deletion2[k+1:]
                            deletions.add(deletion3)

        return deletions

    def correct(self, word):
        """Исправляет слово с опечаткой"""
        # Если слово уже правильное
        if word in self.dictionary:
            return [(word, 0, self.dictionary[word])]

        suggestions = []

        # Генерируем все варианты удалений для входного слова
        possible_deletions = self.generate_deletions(word, self.max_edit_distance)

        # Ищем совпадения в словаре удалений
        for deletion in possible_deletions:
            if deletion in self.deletions_dict:
                suggestions.extend(self.deletions_dict[deletion])

        # Убираем дубликаты и сортируем по расстоянию и частоте
        unique_suggestions = {}
        for word_candidate, distance, freq in suggestions:
            if word_candidate not in unique_suggestions:
                unique_suggestions[word_candidate] = (distance, freq)
            else:
                # Если нашли то же слово с меньшим расстоянием, обновляем
                current_dist, current_freq = unique_suggestions[word_candidate]
                if distance < current_dist or (distance == current_dist and freq > current_freq):
                    unique_suggestions[word_candidate] = (distance, freq)

        # Преобразуем обратно в список и сортируем
        result = []
        for word_candidate, (distance, freq) in unique_suggestions.items():
            result.append((word_candidate, distance, freq))

        # Сортируем сначала по расстоянию, затем по частоте (в убывающем порядке)
        result.sort(key=lambda x: (x[1], -x[2]))

        return result

def load_dictionary(file_path):
    """Загружает словарь из файла"""
    with open(file_path, 'r', encoding='utf-8') as f:
        words = [line.strip().lower() for line in f if line.strip()]
    return words

def evaluate_spell_checker(spell_checker, test_cases):
    """Оценивает качество алгоритма тремя метриками"""
    correct_predictions = 0
    total_predictions = 0
    processing_times = []

    for wrong_word, correct_word in test_cases:
        start_time = time.time()
        suggestions = spell_checker.correct(wrong_word)
        end_time = time.time()

        processing_times.append(end_time - start_time)

        if suggestions:
            total_predictions += 1
            best_suggestion = suggestions[0][0]
            if best_suggestion == correct_word:
                correct_predictions += 1

    # Метрики качества
    accuracy = correct_predictions / len(test_cases) if test_cases else 0
    avg_processing_time = sum(processing_times) / len(processing_times) if processing_times else 0
    coverage = total_predictions / len(test_cases) if test_cases else 0

    return {
        'accuracy': accuracy,
        'avg_processing_time': avg_processing_time,
        'coverage': coverage,
        'total_tested': len(test_cases),
        'correct_predictions': correct_predictions
    }

# Демонстрация работы алгоритма
if __name__ == "__main__":
    # Создаем тестовый словарь
    dictionary_words = [
        'apple', 'application', 'apples', 'banana', 'band', 'bandage',
        'cat', 'category', 'caterpillar', 'dog', 'document', 'doctor',
        'elephant', 'elevator', 'example', 'house', 'hospital', 'hotel'
    ]

    # Создаем тестовые случаи (неправильное слово -> правильное слово)
    test_cases = [
        ('aple', 'apple'),
        ('applicaton', 'application'),
        ('banna', 'banana'),
        ('bdana', 'banana'),
        ('ct', 'cat'),
        ('dgo', 'dog'),
        ('elepant', 'elephant'),
        ('huse', 'house'),
        ('appl', 'apple'),
        ('bnana', 'banana')
    ]

    # Инициализируем и обучаем SymSpell
    symspell = SymSpell(max_edit_distance=2)
    symspell.build_dictionary(dictionary_words)

    # Тестируем на отдельных словах
    test_words = ['aple', 'banna', 'ct', 'dgo', 'elepant', 'xyzabc']

    print("Тестирование SymSpell:")
    print("=" * 50)

    for word in test_words:
        suggestions = symspell.correct(word)
        print(f"Вход: '{word}'")
        if suggestions:
            print(f"Предложения: {[s[0] for s in suggestions[:3]]}")
            print(f"Лучшее: '{suggestions[0][0]}' (расстояние: {suggestions[0][1]}, частота: {suggestions[0][2]})")
        else:
            print("Исправлений не найдено")
        print("-" * 30)

    # Оценка качества
    print("\nОЦЕНКА КАЧЕСТВА:")
    print("=" * 50)

    metrics = evaluate_spell_checker(symspell, test_cases)

    print(f"Точность (Accuracy): {metrics['accuracy']:.3f}")
    print(f"Среднее время обработки: {metrics['avg_processing_time']:.6f} сек")
    print(f"Покрытие (Coverage): {metrics['coverage']:.3f}")
    print(f"Правильных предсказаний: {metrics['correct_predictions']}/{metrics['total_tested']}")

    # Сравнение с реальными данными
    print("\nДЕТАЛЬНЫЕ РЕЗУЛЬТАТЫ:")
    print("=" * 50)

    for wrong, correct in test_cases:
        suggestions = symspell.correct(wrong)
        best_guess = suggestions[0][0] if suggestions else "Нет предложений"
        status = "✓" if best_guess == correct else "✗"
        print(f"{status} '{wrong}' -> '{best_guess}' (ожидалось: '{correct}')")

Тестирование SymSpell:
Вход: 'aple'
Предложения: ['apple', 'apples']
Лучшее: 'apple' (расстояние: 1, частота: 1)
------------------------------
Вход: 'banna'
Предложения: ['banana', 'band']
Лучшее: 'banana' (расстояние: 1, частота: 1)
------------------------------
Вход: 'ct'
Предложения: ['cat']
Лучшее: 'cat' (расстояние: 1, частота: 1)
------------------------------
Вход: 'dgo'
Предложения: ['dog']
Лучшее: 'dog' (расстояние: 1, частота: 1)
------------------------------
Вход: 'elepant'
Предложения: ['elephant']
Лучшее: 'elephant' (расстояние: 1, частота: 1)
------------------------------
Вход: 'xyzabc'
Исправлений не найдено
------------------------------

ОЦЕНКА КАЧЕСТВА:
Точность (Accuracy): 0.900
Среднее время обработки: 0.000032 сек
Покрытие (Coverage): 1.000
Правильных предсказаний: 9/10

ДЕТАЛЬНЫЕ РЕЗУЛЬТАТЫ:
✓ 'aple' -> 'apple' (ожидалось: 'apple')
✓ 'applicaton' -> 'application' (ожидалось: 'application')
✓ 'banna' -> 'banana' (ожидалось: 'banana')
✗ 'bdana' -> 'band' (ожидал

In [ ]:
# Дополнительный тест с большим словарем
def extended_test():
    """Расширенный тест с большим словарем"""
    # Больший словарь для тестирования
    large_dictionary = [
        'the', 'and', 'for', 'are', 'but', 'not', 'you', 'all', 'can', 'had',
        'her', 'was', 'one', 'our', 'out', 'day', 'get', 'has', 'him', 'his',
        'how', 'man', 'new', 'now', 'old', 'see', 'two', 'way', 'who', 'boy',
        'did', 'its', 'let', 'put', 'say', 'she', 'too', 'use', 'any', 'ask',
        'bed', 'big', 'buy', 'can', 'car', 'cup', 'cut', 'dad', 'dog', 'eat',
        'end', 'eye', 'far', 'fly', 'fun', 'get', 'got', 'gum', 'hat', 'hit',
        'hot', 'how', 'ice', 'job', 'key', 'kid', 'law', 'lay', 'leg', 'let',
        'lie', 'lot', 'low', 'mad', 'man', 'may', 'met', 'mrs', 'new', 'nor',
        'not', 'now', 'off', 'old', 'our', 'out', 'own', 'pay', 'put', 'red',
        'run', 'saw', 'say', 'see', 'set', 'she', 'sit', 'son', 'sun', 'tap',
        'tax', 'tea', 'the', 'tie', 'too', 'top', 'try', 'two', 'use', 'war',
        'was', 'way', 'who', 'why', 'win', 'yes', 'yet', 'you', 'able', 'about'
    ]

    # Тестовые случаи с разными типами ошибок
    extended_test_cases = [
        # Ошибки удаления
        ('th', 'the'), ('ad', 'and'), ('fr', 'for'), ('bu', 'but'),
        # Ошибки замены
        ('thee', 'the'), ('ans', 'and'), ('fir', 'for'), ('bit', 'but'),
        # Сложные случаи
        ('yuo', 'you'), ('taht', 'that'), ('wrod', 'word'), ('recieve', 'receive')
    ]

    symspell_large = SymSpell(max_edit_distance=2)
    symspell_large.build_dictionary(large_dictionary)

    print("РАСШИРЕННОЕ ТЕСТИРОВАНИЕ:")
    print("=" * 50)

    metrics = evaluate_spell_checker(symspell_large, extended_test_cases)

    print(f"Точность (Accuracy): {metrics['accuracy']:.3f}")
    print(f"Среднее время обработки: {metrics['avg_processing_time']:.6f} сек")
    print(f"Покрытие (Coverage): {metrics['coverage']:.3f}")
    print(f"Правильных предсказаний: {metrics['correct_predictions']}/{metrics['total_tested']}")

# Запускаем расширенный тест
extended_test()

РАСШИРЕННОЕ ТЕСТИРОВАНИЕ:
Точность (Accuracy): 0.667
Среднее время обработки: 0.000042 сек
Покрытие (Coverage): 0.917
Правильных предсказаний: 8/12


# Задание 3 (2 балла)

Используя любой из алгоритмов из семинара или домашки, детально проанализируйте получаемые ошибки. Улучшите алгоритм так, чтобы исправить ошибки. Улучшения в алгоритме должны быть общими, не привязанными к конкретным словам (например, словарь исключений не будет считаться). За каждое улучшение, которое исправляет 5+ ошибок вы получите 0.5 балла (максимум 2 в целом)